# Is the machine built? Counting ribosomes inside human cells (G14)

This notebook looks at 3D cryo-electron pictures of human cells from the CryoET Data Portal and finds **ribosomes**,
the protein-making machines, by sliding a 3D model of the human ribosome (EMDB EMD-54265) through each picture.

| Test | Question |
|---|---|
| **R1** | Does the finder work? It is tuned on 3 pictures where experts placed ribosomes by hand, then scored once on 3 other hand-labelled pictures. |
| **R2** | In **Jurkat** cells (one of our four screened lines), is the share of ribosomes sitting on membranes close to the share our RNA data predicts should be making ER proteins (0.19)? |

Also reported: ribosome counts in **RPE1** (another of our lines) and HeLa. A scrambled template (same size, no shape)
runs through the same steps as a control.

**How to run**
1. *Runtime → Change runtime type → L4 GPU* (T4 also works, slower).
2. *Runtime → Run all*, and allow Google Drive when asked (results go to *My Drive / cellbrain_cryoet*).
3. Keep the tab open. It downloads about 10 GB of pictures and takes roughly 45–90 minutes.
4. Tell Claude when it is done, or paste the last cell's output.

Fixed version: code commit `a939e1003730`, code id `5c77def2f8d95f82`.

In [ ]:
# Settings
SAVE_TO_DRIVE = True
COMMIT = "a939e10037309da04daf3f49d1f450bdad4d0e3d"
EXPECTED_CODE = "5c77def2f8d95f82"

import torch, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout or "no GPU found")
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > L4 or T4 GPU, then Runtime > Run all again."
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# Library to read the compressed 3D pictures (a few seconds)
!pip -q install "numcodecs>=0.12"
import numcodecs; print("numcodecs", numcodecs.__version__)

In [ ]:
# Download the test code and its fixed run list from GitHub, and check they are the declared version
import os, hashlib, urllib.request
BASE = f"https://raw.githubusercontent.com/Nikku03/cell/{COMMIT}/colab/cryoet/"
os.makedirs("/content/cryoet", exist_ok=True)
for f in ["ribotest.py", "bundle.json"]:
    urllib.request.urlretrieve(BASE + f, "/content/cryoet/" + f)
code_id = hashlib.sha256(open("/content/cryoet/ribotest.py", "rb").read() + open("/content/cryoet/bundle.json", "rb").read()).hexdigest()[:16]
assert code_id == EXPECTED_CODE, f"code id {code_id} is not the declared {EXPECTED_CODE}"
print("code ok:", code_id)

In [ ]:
# Run the test (progress is printed as each picture is searched)
!cd /content/cryoet && python ribotest.py --bundle bundle.json --out /content/ribo_out

In [ ]:
# Save and show the results
import json, shutil, os
res_file = "/content/ribo_out/G14_results.json"
assert os.path.exists(res_file), "the test did not finish; look at the output of the cell above"
if SAVE_TO_DRIVE:
    d = "/content/drive/MyDrive/cellbrain_cryoet"; os.makedirs(d, exist_ok=True); shutil.copy(res_file, d); print("saved to My Drive/cellbrain_cryoet")
from google.colab import files
files.download(res_file)
print("\nIf Claude cannot read your Drive, copy everything below this line into the chat:\n")
print(json.dumps(json.load(open(res_file)), default=float))